# **Feature Audit — company_response_to_consumer**

EDA already confirmed which columns are worth carrying into modeling: `product`, `sub_product`, `issue`, `company`, `submitted_via`, and `state`. This notebook doesn't repeat that analysis — it checks whether those columns are actually *ready* to be encoded, or whether they have quirks (missing values, extreme cardinality, redundancy with each other, unstable low-count categories) that would quietly hurt the model if they went unnoticed.

The output of this notebook is a concrete encoding plan for `05_Feature_Engineering.ipynb` — not more exploratory charts.

In [1]:
import google.colab
google.colab.drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

In [3]:
file_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed/complaints_cleaned.parquet"

df = pd.read_parquet(file_path)

print("Dataset loaded.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded.
Rows: 90112
Columns: 16


We're working with the same cleaned dataset as the EDA notebooks, but from here on we only care about the columns EDA flagged as useful:

In [4]:
candidate_features = ["product", "sub_product", "issue", "company", "submitted_via", "state"]
target = "company_response_to_consumer"

## **1. Data Types and Feature-Level Missingness**

The EDA notebooks focused on missingness in the *target*. Before encoding these columns, we need to check them for their own missing values — a null in `state` or `company` behaves differently in an encoder than a null in the target does, and needs its own handling rule.

In [5]:
feature_audit = pd.DataFrame({
    "dtype": df[candidate_features].dtypes.astype(str),
    "missing_count": df[candidate_features].isna().sum(),
    "missing_pct": (df[candidate_features].isna().mean() * 100).round(2)
})

display(feature_audit)

,dtype,missing_count,missing_pct
product,object,0,0.0
sub_product,object,0,0.0
issue,object,0,0.0
company,object,0,0.0
submitted_via,object,0,0.0
state,object,0,0.0


### Key Finding — Data Types and Missingness

All six candidate features (product, sub_product, issue, company, submitted_via, state)
have 0% missing values — the missingness problem in this dataset is entirely
concentrated in the target, not in these predictors. No "Unknown"/"Missing" category
handling is needed for any of them.

## **2. Cardinality Audit and Encoding Strategy**

Cardinality drives the encoding decision more than anything else. A handful of categories is fine for one-hot encoding; a thousand-plus is not — that's how you end up with a feature matrix wider than the dataset itself. We set simple thresholds and let the data sort itself into a strategy, rather than deciding per-feature by eye.

In [6]:
cardinality = df[candidate_features].nunique().sort_values(ascending=False)
display(cardinality.to_frame(name="unique_categories"))

,unique_categories
company,1499
issue,92
state,60
sub_product,58
product,12
submitted_via,4


In [7]:
def choose_encoding(n_categories):
    if n_categories <= 15:
        return "One-hot"
    elif n_categories <= 100:
        return "Top-N + 'Other' bucket"
    else:
        return "Frequency encoding (too high-cardinality for one-hot or Top-N)"

encoding_plan = cardinality.to_frame(name="unique_categories")
encoding_plan["suggested_encoding"] = encoding_plan["unique_categories"].apply(choose_encoding)

display(encoding_plan)

,unique_categories,suggested_encoding
company,1499,Frequency encoding (too high-cardinality for o...
issue,92,Top-N + 'Other' bucket
state,60,Top-N + 'Other' bucket
sub_product,58,Top-N + 'Other' bucket
product,12,One-hot
submitted_via,4,One-hot


### Key Finding — Cardinality

company (1,499), sub_product (58), and issue (92) match EDA exactly. state came out
at 60 (includes territories like PR, GU, VI, AE, AP, and an "Unknown" bucket), also
consistent with EDA. product, however, came out at 12 — not the ~16 seen in the
original audit notebook. This is expected: 02_Data_Cleaning consolidated overlapping
product categories, so 12 is the correct, current number. Worth a one-line note in
this notebook so nobody re-checking later thinks it's a bug.

## **3. Product vs. Sub-Product Redundancy**

`sub_product` is nested inside `product` — every sub-product belongs to exactly one product. That raises a real question: does `sub_product` carry information *beyond* `product`, or is EDA's sub-product signal (e.g. International money transfer's high untimely rate) really just the parent product's effect showing up at a finer level? If sub_product doesn't add much, keeping both just adds noise and dimensionality for no real gain.

We check this by picking a few products that have several sub-products and looking at how much the untimely-response rate varies *within* each product, across its own sub-products. If sub-products within the same product look similar to each other, sub_product isn't pulling its weight. If they vary a lot, it's earning its place.

In [8]:
df_with_target = df.dropna(subset=[target])

# Pick products with a reasonable number of sub-products and a reasonable complaint count,
# so we're not judging redundancy off a product with only one sub-product to begin with.
subproduct_counts_per_product = (
    df_with_target.groupby("product")["sub_product"]
    .nunique()
    .sort_values(ascending=False)
)

display(subproduct_counts_per_product.head(10))

,sub_product
product,
Debt collection,12
Mortgage,9
"Payday loan, title loan, personal loan, or advance loan",8
"Money transfer, virtual currency, or money service",7
Debt or credit management,4
Checking or savings account,4
Prepaid card,4
Credit reporting or other personal consumer reports,2
Credit card,2


In [9]:
products_to_check = subproduct_counts_per_product.head(5).index

for product_name in products_to_check:
    subset = df_with_target[df_with_target["product"] == product_name]
    untimely_by_subproduct = (
        subset.groupby("sub_product")[target]
        .apply(lambda x: (x == "Untimely response").mean() * 100)
        .round(2)
        .sort_values(ascending=False)
    )
    print(f"Product: {product_name}")
    print(untimely_by_subproduct)
    print("Spread (max - min):", round(untimely_by_subproduct.max() - untimely_by_subproduct.min(), 2))
    print("-" * 50)

Product: Debt collection
sub_product
Federal student loan debt    23.81
Payday loan debt              4.50
Rental debt                   3.59
I do not know                 2.21
Medical debt                  1.10
Credit card debt              0.95
Telecommunications debt       0.88
Other debt                    0.70
Auto debt                     0.44
Credit card                   0.00
Mortgage debt                 0.00
Private student loan debt     0.00
Name: company_response_to_consumer, dtype: float64
Spread (max - min): 23.81
--------------------------------------------------
Product: Mortgage
sub_product
Other type of mortgage                        3.77
Home equity loan or line of credit (HELOC)    3.49
VA mortgage                                   1.53
FHA mortgage                                  1.10
Conventional home mortgage                    0.89
Other mortgage                                0.00
Manufactured home loan                        0.00
Reverse mortgage            

### Key Finding — Product vs. Sub-Product

sub_product adds real signal — it is not redundant with product. Spreads within
product were substantial: Debt collection ranged from 0% to 23.81% untimely across
its sub-products (driven almost entirely by "Federal student loan debt," a striking
outlier worth a manual note), Money transfer ranged up to 34.57% (International money
transfer), and Mortgage showed a smaller but still meaningful 3.77-point spread.
Decision: keep sub_product as its own feature.

## **4. Low-Frequency Categories and Bucketing Rule**

EDA already flagged that some sub-products and states have very few complaints, and that their percentages shouldn't be trusted at face value. Here we turn that caution into an actual rule: any category below a minimum complaint count gets grouped into an `"Other"` bucket at encoding time, instead of getting its own column or its own frequency value.

In [10]:
MIN_CATEGORY_COUNT = 30

def low_frequency_summary(column):
    counts = df[column].value_counts()
    n_rare = (counts < MIN_CATEGORY_COUNT).sum()
    pct_of_rows_affected = (counts[counts < MIN_CATEGORY_COUNT].sum() / len(df) * 100)
    return n_rare, round(pct_of_rows_affected, 2)

bucketing_summary = pd.DataFrame(
    [low_frequency_summary(col) for col in candidate_features],
    index=candidate_features,
    columns=["categories_below_threshold", "pct_of_rows_affected"]
)

display(bucketing_summary)

,categories_below_threshold,pct_of_rows_affected
product,0,0.00
sub_product,16,0.25
issue,31,0.31
company,1372,5.82
submitted_via,0,0.00
state,8,0.09


### Key Finding — Bucketing Rule

At a threshold of 30 complaints: product and submitted_via need no bucketing (0
affected). sub_product, issue, and state are barely affected (0.25%, 0.31%, and 0.09%
of rows respectively) despite having many rare categories — safe to bucket with
negligible data loss. company is the one that matters: 1,372 rare companies affect
5.82% of rows, confirming frequency encoding is the right call, not one-hot.

## **5. Feature-Feature Association (Cramér's V)**

So far every check has been feature-vs-target. This one is feature-vs-feature: do any two of our chosen columns overlap so heavily that they're really carrying the same information twice? `company` and `product` is a natural pair to check — some companies may specialize heavily in one product line, which would make the two partially redundant. We use Cramér's V, a standard measure of association between two categorical variables, scaled between 0 (no association) and 1 (one variable fully determines the other).

In [11]:
def cramers_v(col1, col2):
    contingency = pd.crosstab(col1, col2)
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.sum().sum()
    r, k = contingency.shape
    return np.sqrt((chi2 / n) / (min(r, k) - 1))

In [12]:
feature_pairs_to_check = [
    ("product", "issue"),
    ("product", "sub_product"),
    ("issue", "sub_product"),
    ("company", "product"),
]

association_results = []
for col_a, col_b in feature_pairs_to_check:
    v = cramers_v(df[col_a].dropna(), df[col_b].dropna())
    association_results.append((col_a, col_b, round(v, 3)))

association_df = pd.DataFrame(
    association_results,
    columns=["feature_1", "feature_2", "cramers_v"]
).sort_values("cramers_v", ascending=False)

display(association_df)

,feature_1,feature_2,cramers_v
1,product,sub_product,1.000
0,product,issue,0.957
3,company,product,0.677
2,issue,sub_product,0.531


### Key Finding — Feature Association

product/sub_product shows Cramér's V = 1.000 — expected, since sub_product is
structurally nested inside product. More important: product/issue = 0.957 — nearly
as strong, and NOT structurally guaranteed the way the sub_product relationship is.
This means product and issue carry almost the same information. company/product =
0.677 (moderate — some companies specialize in certain products). issue/sub_product
= 0.531 (moderate).

Decision: keep both product and issue for now, since tree-based models tolerate
correlated categorical features reasonably well — but flag this for revisiting if
feature importance in modeling shows one is doing all the work for both.

In [13]:
def apply_other_bucket(series, min_count=30):
    counts = series.value_counts()
    rare_categories = counts[counts < min_count].index
    return series.where(~series.isin(rare_categories), "Other")

# Quick sanity check on the feature with the most rare categories
company_bucketed = apply_other_bucket(df["company"])
print("Categories before bucketing:", df["company"].nunique())
print("Categories after bucketing:", company_bucketed.nunique())
print(company_bucketed.value_counts().head(10))

Categories before bucketing: 1499
Categories after bucketing: 128
company
Experian Information Solutions Inc.       21764
TRANSUNION INTERMEDIATE HOLDINGS, INC.    19214
EQUIFAX, INC.                             19050
Other                                      5242
Pending Company Match                      3207
BANK OF AMERICA, NATIONAL ASSOCIATION      1139
CAPITAL ONE FINANCIAL CORPORATION          1032
WELLS FARGO & COMPANY                      1005
CITIBANK, N.A.                              966
JPMORGAN CHASE & CO.                        794
Name: count, dtype: int64


### Key Finding — Bucketing Preview

Bucketing collapses company from 1,499 to 128 categories, dominated by the three
credit bureaus (Experian, TransUnion, Equifax) — consistent with credit reporting
being the largest product category. One anomaly: "Pending Company Match" appears as
the 5th-largest category (3,207 complaints, 3.6%) — this is very likely a placeholder
for complaints where CFPB's company-matching process failed, not a real company.
Recommendation: treat "Pending Company Match" as its own explicit "Unknown company"
category rather than letting it blend in as if it were a real company's response
pattern — it may behave differently and shouldn't be interpreted as one company's
actual performance.

## **6. Final Encoding Spec — Handoff to Feature Engineering**

| Feature | Cardinality | Encoding | Bucketing rule | Keep? |
|---|---|---|---|---|
| `product` | 12 | One-hot | None needed | Yes |
| `sub_product` | 58 | Top-N + Other | Categories under 30 complaints → Other | Yes — confirmed in Section 3, adds real signal (spreads up to 34.57 points within product) |
| `issue` | 92 | Top-N + Other | Categories under 30 complaints → Other | Yes — but noted as highly associated with product (Cramér's V = 0.957), revisit if feature importance later shows redundancy |
| `company` | 1,499 (→ 128 after bucketing) | Frequency encoding | Categories under 30 complaints → Other | Yes — flag "Pending Company Match" as an explicit unknown-company category, not a real company |
| `submitted_via` | 4 | One-hot | None needed | Yes |
| `state` | 60 | Top-N + Other | Categories under 30 complaints → Other | Yes — weak signal, low priority |

**Carried over from EDA:**
- `timely_response` is dropped entirely — confirmed leakage.
- Rows where the target is missing or `In progress` are excluded before this table is even applied.

**Next notebook:** `05_Feature_Engineering.ipynb` — apply this encoding spec, build the model-ready dataset, and save it separately from the cleaned data.